# a 数据清洗与图表分析

读取 `a/国债ETF数据/3秒快照` 下的 ETF 3 秒快照数据。当前清洗口径：删除导出残留索引列，统一时间戳，保留所有交易阶段，把价格字段里的无效 `0` 先转成 `NaN`，暂时不填补也不删除。

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == 'notebooks' else cwd
A_ROOT = PROJECT_ROOT / 'a' / '国债ETF数据' / '3秒快照'
raw_files = sorted(A_ROOT.rglob('*snapshot.csv'))
pd.DataFrame({'file': [str(p.relative_to(PROJECT_ROOT)) for p in raw_files], 'size_mb': [round(p.stat().st_size/1024/1024, 2) for p in raw_files]})

## 读取与轻量清洗

注意：`volume/amount/num_trades` 的 0 暂时保留；价格、盘口价、IOPV 的 0 先标成 NaN。

In [ ]:
PRICE_COLS = ['pre_close','last','open','high','low','close','high_limited','low_limited',
              'ask_price1','bid_price1','ask_price2','bid_price2','ask_price3','bid_price3',
              'ask_price4','bid_price4','ask_price5','bid_price5']
VOLUME_COLS = ['volume','amount','num_trades',
               'ask_volume1','bid_volume1','ask_volume2','bid_volume2',
               'ask_volume3','bid_volume3','ask_volume4','bid_volume4',
               'ask_volume5','bid_volume5']

def read_one(path):
    df = pd.read_csv(path, low_memory=False)
    df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed')]
    df = df.loc[:, df.columns.astype(str).str.strip() != '']
    df.columns = df.columns.astype(str).str.strip()
    if 'iopv' in df.columns:
        df = df.drop(columns=['iopv'])
    df['source_file'] = path.name
    df['symbol_dir']  = path.parent.name
    df['code']               = df['code'].astype('string').str.strip()
    df['trading_phase_code'] = df['trading_phase_code'].astype('string').str.strip()
    df['trade_time'] = pd.to_datetime(df['trade_time'], errors='coerce')
    for c in PRICE_COLS + VOLUME_COLS:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
    cols = [c for c in PRICE_COLS if c in df.columns]
    df[cols] = df[cols].replace(0, np.nan)
    df = df.drop_duplicates(subset=['code', 'trade_time'], keep='first')
    df = df[df['last'].notna()].copy()   # drop S 10 / C111 (all-zero-price phases)
    df['date']          = df['trade_time'].dt.date
    df['is_continuous'] = df['trading_phase_code'].eq('T111')
    return df

def read_symbol(symbol):
    fs = [p for p in raw_files if p.parent.name == symbol]
    df = pd.concat([read_one(p) for p in fs], ignore_index=True)
    return df.sort_values(['code','trade_time','source_file']).reset_index(drop=True)

etf_all = pd.concat([read_symbol('511090'), read_symbol('511130')], ignore_index=True)
etf_all.shape, etf_all.head()


In [ ]:
def make_ohlcv(df, freq='1D', phase='T111'):
    '''
    3 秒快照 → 任意时间粒度 OHLCV。
    freq : 'raw' 返回原始快照行（含增量列 vol_inc / amt_inc）
           其余传 pandas offset alias，如 '1min' '5min' '15min' '30min' '1h' '1D'
    phase: 'T111'（默认）| 'E110' | None（保留所有剩余行）
    '''
    g = df.copy() if phase is None else df[df['trading_phase_code'] == phase].copy()
    g = g.sort_values(['code', 'trade_time'])
    # cumulative intraday volume → per-snapshot increment
    g['vol_inc'] = (g.groupby(['code', 'date'])['volume']
                     .transform(lambda s: s.diff().clip(lower=0).fillna(s)))
    g['amt_inc'] = (g.groupby(['code', 'date'])['amount']
                     .transform(lambda s: s.diff().clip(lower=0).fillna(s)))
    if freq == 'raw':
        return g.reset_index(drop=True)
    ohlcv = (
        g.set_index('trade_time')
        .groupby('code')
        .resample(freq)
        .agg(open=('last','first'), high=('last','max'),
             low=('last','min'),  close=('last','last'),
             volume=('vol_inc','sum'), amount=('amt_inc','sum'),
             rows=('last','size'))
        .reset_index()
        .dropna(subset=['open','high','low','close'], how='all')
    )
    ohlcv['return']     = ohlcv.groupby('code')['close'].pct_change()
    ohlcv['cum_return'] = (
        (1 + ohlcv['return'].fillna(0)).groupby(ohlcv['code']).cumprod() - 1
    )
    return ohlcv


## 各字段无效值（NaN / 无效 0）统计

In [ ]:
from collections import defaultdict

price_invalid = defaultdict(int)
vol_zero      = defaultdict(int)

# 直接从原始 CSV 统计，避免受清洗时 drop 行的影响
for p in raw_files:
    df_raw = pd.read_csv(p, low_memory=False)
    df_raw.columns = df_raw.columns.astype(str).str.strip()
    for c in PRICE_COLS:
        if c in df_raw.columns:
            s = pd.to_numeric(df_raw[c], errors='coerce')
            price_invalid[c] += s.isna().sum() + (s == 0).sum()
    for c in VOLUME_COLS:
        if c in df_raw.columns:
            s = pd.to_numeric(df_raw[c], errors='coerce')
            vol_zero[c] += (s == 0).sum()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 价格字段：NaN + 无效 0
pd.Series(price_invalid).sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('价格字段无效值数量（原始数据：NaN + 0）')
axes[0].set_ylabel('数量')
axes[0].tick_params(axis='x', rotation=45)

# 成交量字段：0 值
pd.Series(vol_zero).sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('成交量字段 0 值数量（原始数据）')
axes[1].set_ylabel('数量')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()


## 质量检查

In [ ]:
quality = etf_all.groupby('code').agg(rows=('code','size'), start=('trade_time','min'), end=('trade_time','max'), last_nan_rate=('last', lambda s: s.isna().mean()), bid1_nan_rate=('bid_price1', lambda s: s.isna().mean()), ask1_nan_rate=('ask_price1', lambda s: s.isna().mean()))
dup = etf_all.duplicated(['code','trade_time']).groupby(etf_all['code']).sum().rename('duplicate_code_time')
display(quality.join(dup))
phase_table = pd.crosstab(etf_all['code'], etf_all['trading_phase_code'])
display(phase_table)
checks = etf_all.assign(bid_gt_ask=etf_all['bid_price1'].notna() & etf_all['ask_price1'].notna() & (etf_all['bid_price1'] > etf_all['ask_price1']), last_outside_limit=etf_all['last'].notna() & etf_all['high_limited'].notna() & etf_all['low_limited'].notna() & ((etf_all['last'] > etf_all['high_limited']) | (etf_all['last'] < etf_all['low_limited'])))
checks.groupby('code')[['bid_gt_ask','last_outside_limit']].sum()

## 零值与 NaN 诊断

In [ ]:
# 按交易阶段统计各价格字段的 0 率和 NaN 率（清洗前 vs 清洗后）
phase_stats = []
for phase, g in etf_all.groupby('trading_phase_code'):
    row = {'phase': phase, 'rows': len(g)}
    for c in ['last', 'open', 'high', 'low', 'close', 'ask_price1', 'bid_price1', 'iopv', 'volume', 'amount']:
        if c in g.columns:
            row[f'{c}_nan%'] = round(g[c].isna().mean() * 100, 1)
    phase_stats.append(row)
phase_df = pd.DataFrame(phase_stats).set_index('phase')
display(phase_df)

# 结论表
print("""
字段/交易阶段 分析：
  S 10  : last/bid/ask/iopv 全为 NaN（原始 0→NaN），volume=0。无价格信息，不适合任何价格分析。
  C111  : last/bid/ask 全为 NaN，close 在 E110 之后才有值。volume 是收盘竞价累计值，可参考但非必需。
  T111  : last/open/high/low/bid1/ask1 零 NaN，数据干净。
           close 字段 100% NaN（原始全为 0）—— 这是正常的：交易所在收盘竞价结束后才回填 close，
           日内快照中 close 永远为 0，不要用它做日 K，应用 last 字段做 OHLC（notebook 已正确处理）。
           iopv 100% NaN（原始全为 0）—— 该数据源未提供 iopv，忽略此字段。
  E110  : last/close 有效（close 在此阶段已回填为当日收盘价）。
           如需精确收盘价可从 E110 末行取 close，而不是从 T111 末行取 last。

volume/amount/num_trades：T111 内为日内累计值（单调递增，跨日归零），'last' 聚合即日末成交量，正确。

需要 drop 的情况：
  - 做价格分析（折线、K线、RSI、布林带）：只保留 T111（is_continuous=True），已在 notebook 中处理。
  - S 10 / C111 行的价格字段全为 NaN，保留在 nan_marked 文件中无害，但不要参与任何价格计算。
  - iopv 字段可以整列删除，全为 NaN 无信息价值。
""")

## OHLCV 聚合（频率可调）

In [ ]:
# ── 时间粒度 ── 修改此处即可切换所有图表
# 可选: 'raw'(3秒快照), '1min', '5min', '15min', '30min', '1h', '1D'
FREQ = '1D'

ohlcv = make_ohlcv(etf_all, freq=FREQ)
print(f'freq={FREQ}  shape={ohlcv.shape}')
ohlcv.head()


## 交易日历与缺失标注（假期/春节）

生成完整日历，找出非周末的缺失日期（如春节休市），并在日频 OHLCV 中显式占位为 NaN，避免后续预测序列把『跳日』当成连续时间。

> 若切换到分钟/小时粒度（FREQ != '1D'），通常只需保留交易时段内的时间戳，无需插缺失。

In [ ]:
# 1) 基于原始清洗数据生成每只 code 的实际交易日期
trading_dates_by_code = (
    etf_all.groupby('code')['trade_time']
    .apply(lambda s: pd.to_datetime(s.dt.date.dropna().unique()))
)

# 2) 若当前是日频，补全日历并标注缺失
if FREQ == '1D':
    ohlcv_list = []
    for code, g in ohlcv.groupby('code'):
        g = g.set_index('trade_time').sort_index()
        idx = pd.date_range(start=g.index.min(), end=g.index.max(), freq='D')
        g = g.reindex(idx)
        g['code'] = code
        g['is_trading_day'] = g.index.isin(trading_dates_by_code.get(code, []))
        g.index.name = 'trade_time'
        ohlcv_list.append(g.reset_index())
    ohlcv = pd.concat(ohlcv_list, ignore_index=True)

    # 打印每只 code 的缺失日期（非周末）
    for code in ohlcv['code'].unique():
        sub = ohlcv[ohlcv['code'] == code]
        missing = sub.loc[~sub['is_trading_day'], 'trade_time']
        # 只保留工作日（Mon-Fri）
        missing = missing[missing.dt.dayofweek < 5]
        print(f"{code} 缺失的非周末日期 ({len(missing)} 天):", missing.tolist())

    display(ohlcv[~ohlcv['is_trading_day']].head())
else:
    print("当前非日频，不做日历插值。请确保模型输入时按交易时间索引即可。")


## 图 1-4：折线图、K 线图、柱状图、多折线图

In [ ]:
# 1 折线图
for code, g in ohlcv.groupby('code'):
    plt.figure(figsize=(14, 6))
    plt.plot(g['trade_time'], g['close'], marker='o', markersize=2, linewidth=1.4, label=code)
    plt.title(f'{code} 收盘价趋势 (freq={FREQ})'); plt.xlabel('时间'); plt.ylabel('价格'); plt.legend(); plt.show()

# 2 K 线图
def plot_candles(df, code, last_n=120):
    g = df[df['code'].eq(code)].dropna(subset=['open','high','low','close']).tail(last_n).copy()
    x = np.arange(len(g)); fig, ax = plt.subplots(figsize=(15, 6))
    for i, r in enumerate(g.itertuples()):
        color = '#d62728' if r.close >= r.open else '#2ca02c'
        ax.vlines(i, r.low, r.high, color=color, linewidth=1)
        ax.bar(i, max(abs(r.close - r.open), 1e-4), bottom=min(r.open, r.close),
               width=0.65, color=color, alpha=0.75)
    step = max(1, len(g) // 10)
    ax.set_xticks(x[::step])
    ax.set_xticklabels(g['trade_time'].dt.strftime('%Y-%m-%d %H:%M').iloc[::step],
                       rotation=45, ha='right')
    ax.set_title(f'{code} K 线 (freq={FREQ})'); ax.set_ylabel('价格'); plt.show()
for code in ohlcv['code'].unique(): plot_candles(ohlcv, code)

# 3 柱状图：保留的交易阶段 & 每日快照量
phase_table = pd.crosstab(etf_all['code'], etf_all['trading_phase_code'])
for code in phase_table.index:
    phase_table.loc[[code]].T.plot(kind='bar', figsize=(10, 5), title=f'{code} 保留交易阶段的数据量')
    plt.xlabel('交易阶段'); plt.ylabel('行数'); plt.show()
daily_counts = etf_all.groupby(['code', etf_all['trade_time'].dt.date]).size().unstack(0)
for code in daily_counts.columns:
    daily_counts[[code]].plot(kind='bar', figsize=(16, 5), width=0.85, title=f'{code} 每日快照数量')
    plt.xlabel('日期'); plt.ylabel('快照行数'); plt.show()

# 4 多折线图
price_wide = ohlcv.pivot(index='trade_time', columns='code', values='close')
for code in price_wide.columns:
    price_wide[[code]].plot(figsize=(14, 6), linewidth=1.8, title=f'{code} 收盘价走势 (freq={FREQ})')
    plt.xlabel('时间'); plt.ylabel('价格'); plt.show()
for code in price_wide.columns:
    normalized = price_wide[[code]] / price_wide[code].ffill().bfill().iloc[0]
    normalized.plot(figsize=(14, 6), linewidth=1.8, title=f'{code} 归一化价格走势，首期=1 (freq={FREQ})')
    plt.xlabel('时间'); plt.ylabel('归一化价格'); plt.show()


## 图 5-9：折柱图、阈值折线图、布林带、收益率直方图、累计收益率

In [ ]:
# 5 折柱图：价格 + 成交量
for code, g in ohlcv.groupby('code'):
    fig, ax1 = plt.subplots(figsize=(15, 6)); ax2 = ax1.twinx()
    ax1.plot(g['trade_time'], g['close'], color='#1f77b4', linewidth=1.8)
    ax2.bar(g['trade_time'], g['volume'], color='#888888', alpha=0.28)
    ax1.set_title(f'{code} 价格 + 成交量 (freq={FREQ})')
    ax1.set_ylabel('价格'); ax2.set_ylabel('成交量'); plt.show()

# 6 RSI
def add_rsi(g, window=14):
    g = g.sort_values('trade_time').copy(); d = g['close'].diff()
    gain = d.clip(lower=0).rolling(window).mean()
    loss = (-d.clip(upper=0)).rolling(window).mean()
    g['rsi'] = 100 - 100 / (1 + gain / loss); return g
ohlcv_rsi = pd.concat([add_rsi(g) for _, g in ohlcv.groupby('code')], ignore_index=True)
for code, g in ohlcv_rsi.groupby('code'):
    plt.figure(figsize=(14, 4)); plt.plot(g['trade_time'], g['rsi'], linewidth=1.2)
    plt.axhline(70, color='red',   linestyle='--', label='超买 70')
    plt.axhline(30, color='green', linestyle='--', label='超卖 30')
    plt.ylim(0, 100); plt.title(f'{code} RSI (freq={FREQ})')
    plt.xlabel('时间'); plt.ylabel('RSI'); plt.legend(); plt.show()

# 7 布林带
for code, g in ohlcv.groupby('code'):
    g = g.sort_values('trade_time').copy()
    mid = g['close'].rolling(20).mean(); std = g['close'].rolling(20).std()
    upper, lower = mid + 2*std, mid - 2*std
    plt.figure(figsize=(14, 6))
    plt.plot(g['trade_time'], g['close'], label='close')
    plt.plot(g['trade_time'], mid, label='20-period MA')
    plt.plot(g['trade_time'], upper, '--', label='upper')
    plt.plot(g['trade_time'], lower, '--', label='lower')
    plt.fill_between(g['trade_time'], lower, upper, alpha=0.12)
    plt.title(f'{code} 布林带 (freq={FREQ})'); plt.xlabel('时间'); plt.ylabel('价格'); plt.legend(); plt.show()

# 8 收益率直方图
for code, g in ohlcv.groupby('code'):
    r = g['return'].dropna()
    plt.figure(figsize=(10, 5)); plt.hist(r, bins=30, alpha=0.75, edgecolor='white')
    plt.axvline(r.mean(), color='red', linestyle='--', label=f'mean={r.mean():.6f}')
    plt.title(f'{code} 收益率直方图 (freq={FREQ})'); plt.xlabel('收益率'); plt.ylabel('频数')
    plt.legend(); plt.show()

# 9 累计收益率折线图
cum_ret_wide = ohlcv.pivot(index='trade_time', columns='code', values='cum_return')
for code in cum_ret_wide.columns:
    cum_ret_wide[[code]].plot(figsize=(14, 6), linewidth=1.8, title=f'{code} 累计收益率 (freq={FREQ})')
    plt.axhline(0, color='black', linewidth=0.8); plt.xlabel('时间'); plt.ylabel('累计收益率'); plt.show()


## 保存 NaN 标记版

这一步保存轻量清洗结果：保留 NaN，不填补，不先筛掉交易阶段。

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'a_nan_marked'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_cols = [c for c in etf_all.columns if c != 'date']
for code, g in etf_all.groupby('code'):
    short = code.replace('.SH', '')
    start = g['trade_time'].min().strftime('%Y%m%d'); end = g['trade_time'].max().strftime('%Y%m%d')
    out = OUTPUT_DIR / f'{short}_{start}_{end}_nan_marked.CSV'
    g[save_cols].to_csv(out, index=False, encoding='utf-8-sig')
    print(out)